In [54]:
import pathlib
import numpy as np
from astropy.table import Table
import matplotlib 

font = {'family' : 'arial',
        'weight' : 'bold',
        'size'   : 22}

matplotlib.rc('font', **font)

import matplotlib.pyplot as plt
%matplotlib inline
import galsim
import fitsio

from kl_tools.velocity import VelocityMap
from kl_tools.kross.kross_utils import theta2pars, pars2theta
from kl_tools.utils import build_map_grid, get_base_dir, make_dir, MidpointNormalize


data_path = "../../data/kross/"

In [62]:
import fitsio
from pathlib import Path
from glob import glob
from astropy.coordinates import SkyCoord
from astropy.units import deg

def get_kross_obj_data(kid, vb=False):
    kross_dir = get_base_dir() / 'data/kross'
    cosmo_dir = get_base_dir() / 'data/cosmos'
    kross_data = fitsio.read(kross_dir / 'kross_release_v2.fits', ext=1)

    # need all the coords for a match to the COSMOS cutouts
    kross_ra = kross_data['RA']
    kross_dec = kross_data['DEC']
    kross_names = kross_data['NAME']
    kross_coords = SkyCoord(ra=kross_ra*deg, dec=kross_dec*deg)

    # clean up the names
    for i, name in enumerate(kross_names):
        new = name.strip()
        kross_names[i] = new

    # get the KROSS sky coordinates to match to the COSMOS cutouts
    cosmo_files = glob(str(cosmo_dir / 'cutouts/*.fits'))
    cosmo_cutouts = {}
    for f in cosmo_files:
        fname = Path(f).name

        # match to the KROSS sky coordinates to get the obj name
        ra  = float(fname.split('_')[1])
        dec = float(fname.split('_')[2])
        coord = SkyCoord(ra=ra*deg, dec=dec*deg)

        indx, sep, _ = coord.match_to_catalog_sky(kross_coords)
        cosmo_cutouts[kross_names[indx]] = f

    row = kross_data[kross_data['KID'] == kid] 
    name = row['NAME'][0].strip()
    ra = kross_data['RA']
    dec = kross_data['DEC']

    cube_file = kross_dir / 'cubes' / f'{name}.fits'
    velocity_file = kross_dir / 'vmaps' / f'{name}.fits' 
    sigma_file = kross_dir / 'disp' / f'{name}.fits' 
    halpha_file = kross_dir / 'halpha' / f'{name}.fits'
    hst_file = cosmo_cutouts[name]

    # grab all of the object data
    try:
        cube, cube_hdr = fitsio.read(cube_file, header=True)
    except FileNotFoundError:
        if vb is True:
            print(f'Cube file not found: {cube_file}')
        cube, cube_hdr = None
    try:
        velocity, velocity_hdr = fitsio.read(velocity_file, header=True)
    except FileNotFoundError:
        if vb is True:
            print(f'Velocity map file not found: {velocity_file}')
        velocity, velocity_hdr = None, None
    try:
        sigma, sigma_hdr = fitsio.read(sigma_file, header=True)
    except FileNotFoundError:
        if vb is True:
            print(f'Sigma file not found: {sigma_file}')
        sigma, sigma_hdr = None, None
    try:
        halpha, halpha_hdr = fitsio.read(halpha_file, header=True)
    except FileNotFoundError:
        if vb is True:
            print(f'Halpha file not found: {halpha_file}')
        halpha, halpha_hdr = None, None
    try:
        hst, hst_hdr = fitsio.read(hst_file, header=True)
    except FileNotFoundError:
        if vb is True:
            print(f'HST file not found: {hst_file}')
        hst, hst_hdr = None, None

    obj_data = {
        'catalog': row,
        'cube': cube,
        'cube_hdr': cube_hdr,
        'velocity': velocity,
        'velocity_hdr': velocity_hdr,
        'sigma': sigma,
        'sigma_hdr': sigma_hdr,
        'halpha': halpha,
        'halpha_hdr': halpha_hdr,
        'hst': hst,
        'hst_hdr': hst_hdr
    }

    return obj_data

kid = 171
obj = get_kross_obj_data(kid)
print(obj['catalog'])
print(obj['catalog']['KID'])
print(obj['cube'].shape)
print(obj['velocity'].shape)
print(obj['sigma'].shape)
print(obj['halpha'].shape)
print(obj['hst'].shape)

[(171, 'C-zcos_z1_925', 150.23446, 2.3113942, 20.889421, 22.346275, 21.3459167, -22.49, 1.51715547e+10, 0., 'I  ', 1, 57.080894, 3.7296395, 1.118892, 0., 0.71416855, 45.59468152, 14.118937, 0, 0.9128704, 9.80359196e-17, 6.13292365e+41, 14.023154, 102.114235, 0, 0, 56., 108.9014, 164.16039, 113.262596, 169.30264, 29.302696, 62.375507, 0, 'RT+', 751.4111, 309.69632, 424.84125, 0., 24.5511322, 2.1162622, 23.20235443, 2., 'O', 0.65742871, 0.19722862, 0.64130266, 0.36181087, 0.29563903)]
[171]
(2048, 30, 30)
(45, 45)
(45, 45)
(45, 45)
(168, 168)


In [121]:
# Set up the velocity model to fit.
# Steal liberally from Spencer's model-fitting code.
def get_hst_wcs(header):
    hdrdict = dict(header)
    keys = ['SIMPLE','BITPIX','NAXIS','NAXIS1','NAXIS2','EXTEND','CTYPE1','CTYPE2','CRVAL1','CRVAL2','CRPIX1','CRPIX2','CDELT1','CDELT2','CROTA2','CD1_1','CD2_1','CD1_2','CD2_2']
    wcsdict = {}
    for ikey in keys:
        wcsdict[ikey] = hdrdict[ikey]
    wcs,origin = galsim.wcs.readFromFitsHeader(wcsdict)
    return wcs,origin


def get_imodel(pars, scale_ratio = 0.2, psf= None, image=None):
    # image pars:
    
    x,y = pars[0],pars[1]
    flux = pars[2]
    hlr = pars[3]
    inclination = pars[4]*galsim.radians
    orientation = pars[5]*galsim.radians
    scale_h_over_r = scale_ratio
    model = galsim.InclinedExponential(inclination=inclination,half_light_radius=hlr,flux=flux,scale_h_over_r=scale_h_over_r)
    model.rotate(orientation)
    psfConvModel = galsim.Convolve([model, psf])
    image_model = psfConvModel.drawImage(image=image,center=galsim.positionD(x,y))    
    return image_model.array

def get_vmodel(pars, X, Y):
    vmap = VelocityMap('offset', pars)
    return vmap('obs', X, Y)

def vresiduals(theta, data, weights, mask, X, Y):
    pars = theta2pars(theta)
    model = get_model(pars, X, Y)
    res = ((data - model) * np.sqrt(weights) * mask).flatten()
    return res

def iresiduals(theta, image, weights, mask, p.,msf):
    imodel = get_imodel(theta, psf=psf,image=image)
    res = ((image.array - imodel) * np.sqrt(weights) * mask).flatten()
    return res

def get_residuals(parameters, imdata, specdata, vweight = 0.5):

    specpars = parameters[:specdata['npars']]
    impars = parameters[specdata['npars']:]
    vresid = vresiduals(specpars,specdata['data'],specdata['weights'],specdata['mask'],specdata['X'],specdata['Y'])
    iresid = iresiduals(impars, imdata['image'], imdata['weights'],imdata['mask'], imdata['psf'])
    resid = np.concatenate([(1-vweight)*vresid,vwieght*vresid])
    return resid
    



In [130]:
image_meta = {}
imgwcs,origin = get_hst_wcs(obj['hst_hdr'])
image_meta['image'] = galsim.Image(obj['hst'],wcs=imgwcs)

In [119]:
from astropy import wcs
#galsim.wcs.readFromFitsHeader(dict(obj['hst_hdr']))
wcsdefault = wcs.WCS()
type(obj['hst_hdr'])
hdrdict = dict(obj['hst_hdr'])

wcs = wcs.WCS(wcsdict)
galsim.wcs.readFromFitsHeader(wcsdict)

(galsim.GSFitsWCS(_data = ['TAN', array([14250.2, -13165.333]), array([[-8.333333e-06, 0.0], [0.0, 8.333333e-06]]), coord.CelestialCoord(coord.Angle(2.6200240676666944, coord.radians), coord.Angle(0.03841422729046653, coord.radians)), None, None, None]),
 galsim.PositionI(x=1, y=1))